# Classificador de Integridade do Universo — v2 (critério refinado)

**Apêndice — TCC (otimização de carteiras, B3).** Exclusão **estática** do universo final, por critério objetivo, sobre o **COTAHIST tratado**.

A v1 (mínimo anual < R$ 1) gerou falsos positivos graves (Santander, Gerdau, Usiminas, Kepler, Unipar — preço nominal baixo ≠ penny). A v2 corrige com três princípios:

1. **Penny por predominância + iliquidez.** Exclui por preço só se a **mediana anual** de `PRECO_UNIT` < R$ 1 **e** a liquidez (ADTV) estiver **abaixo de um percentil** do universo. Preço baixo de empresa líquida (Santander) não é penny.
2. **Continuidade de identidade.** Empresa que se recuperou **sob o mesmo ticker/ISIN** (ex.: Eneva) tem problema **antigo desconsiderado** — *preserva recuperadas*. Empresa cuja equity foi zerada e **reemitida com ISIN/ticker novo** (reorganização: Gol `GOLL4`→`GOLL54`, Fictor `ATOM3`→`FICT3`) é **excluída** — a continuidade econômica rompeu. Essa distinção vem direto da descontinuidade de ISIN observada nos dados.
3. **Situação especial recente.** `FLAG_SITUACAO` (RJ/concordata/RAET/intervenção) exclui se ocorrer nos **anos recentes**; RJ antiga já superada não exclui (consistente com *preserva recuperadas*).

> Os reorganizados (`REORG_TICKERS`) são excluídos por **identidade rompida**; preço/RJ entram como corroboração no motivo. Os retornos seguem na série ajustada da Economática; o COTAHIST define apenas a lista de exclusão.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================ CONFIGURAÇÃO ============================
PASTA_COTAHIST = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\B3_COTAHIST_Tratatos")

# ANTES (lê o universo errado — a lista pós-integridade de 109, que já tirou os 9 mas mantém AMER3/OIBR3/...):
# ARQ_UNIVERSO = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\Tickers\tickers_finais.csv")

# DEPOIS (lê o universo pré-integridade de 118, exportado pelo NB03 na Edição 1):
ARQ_UNIVERSO = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\Tickers\universo_pos_liquidez.csv")

PASTA_SAIDA    = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\Tickers")

ANO_INI, ANO_FIM = 2010, 2025
PISO_PRECO       = 1.00              # piso de penny (R$) sobre a MEDIANA anual de PRECO_UNIT
PERCENTIL_ADTV   = 20                # liquidez abaixo deste percentil do universo = "ilíquida"
ANOS_RECENTES    = [2023, 2024, 2025]  # janela "recente" p/ penny e situação especial (identidade contínua)

# Reorganizações (identidade rompida -> exclusão). Edite conforme a Seção 2 sugerir novos ⚠.
REORG_TICKERS = {"FICT3", "GOLL54", "VSTE3", "PDTC3"}

# Mapa de linhagem: combina histórico do predecessor (reorg E renomeações, p/ cobertura de dados).
MAPA_LINHAGEM = {
    "FICT3": ["ATOM3"], "GOLL54": ["GOLL4"], "VSTE3": ["LLIS3"], "PDTC3": ["IDNT3"],   # reorg
    "AXIA3": ["ELET3"], "AXIA6": ["ELET6"], "EMBJ3": ["EMBR3"], "RIAA3": ["GUAR3"],     # renomeações saudáveis (cobertura)
}
DETECTAR_LINHAGEM = True
CONTINUIDADE_DIAS = 7
# =====================================================================

## 1. Resumos por ticker (preço, liquidez e situação especial)

In [2]:
def carregar_universo(arq):
    u = pd.read_csv(arq); col = 'ticker' if 'ticker' in u.columns else u.columns[0]
    return u[col].astype(str).str.strip().tolist()

def ler_ticker(pasta):
    nome = pasta.name; pq, csv = pasta / f"{nome}.parquet", pasta / f"{nome}.csv"
    cols = ['DATA_PREGAO', 'ANO', 'PRECO_UNIT', 'VOLTOT', 'FLAG_SITUACAO', 'CODBDI']
    if pq.exists():
        d = pd.read_parquet(pq)
    elif csv.exists():
        d = pd.read_csv(csv, parse_dates=['DATA_PREGAO'])
    else:
        return None
    for c in cols:
        if c not in d.columns: d[c] = np.nan
    d['ANO'] = pd.to_numeric(d['ANO'], errors='coerce').astype('Int64')
    d['PRECO_UNIT'] = pd.to_numeric(d['PRECO_UNIT'], errors='coerce')
    d['VOLTOT'] = pd.to_numeric(d['VOLTOT'], errors='coerce')
    d['FLAG_SITUACAO'] = d['FLAG_SITUACAO'].astype(bool)
    return d

def resumir(d):
    dd = d[d['ANO'].between(ANO_INI, ANO_FIM)]
    med = {int(a): float(v) for a, v in dd.groupby('ANO')['PRECO_UNIT'].median().items() if pd.notna(v)}
    vol = {int(a): float(v) for a, v in dd.groupby('ANO')['VOLTOT'].median().items() if pd.notna(v)}
    sit = dd[dd['FLAG_SITUACAO']]
    return {'primeiro': dd['DATA_PREGAO'].min(), 'ultimo': dd['DATA_PREGAO'].max(),
            'med_por_ano': med, 'volmed_por_ano': vol,
            'volmed_global': float(dd['VOLTOT'].median()) if len(dd) else np.nan,
            'anos_sit': sorted({int(a) for a in sit['ANO'].dropna().unique()}),
            'codbdi_sit': sorted(sit['CODBDI'].dropna().astype(str).unique().tolist())}

if not PASTA_COTAHIST.exists():
    raise FileNotFoundError(f"Pasta COTAHIST não encontrada: {PASTA_COTAHIST}")
universo = carregar_universo(ARQ_UNIVERSO)
pastas = [p for p in PASTA_COTAHIST.iterdir() if p.is_dir()]
print(f"Universo: {len(universo)} | lendo {len(pastas)} tickers do COTAHIST...")
RESUMO = {}
for i, p in enumerate(pastas, 1):
    d = ler_ticker(p)
    if d is not None and len(d): RESUMO[p.name] = resumir(d)
    if i % 200 == 0: print(f"  {i}/{len(pastas)}")
print(f"Resumos: {len(RESUMO)}")

Universo: 118 | lendo 893 tickers do COTAHIST...


  200/893


  400/893


  600/893


  800/893


Resumos: 893


## 2. Sugestão de predecessores (revise os ⚠ e atualize `REORG_TICKERS`/`MAPA_LINHAGEM`)

Regra de bolso: o predecessor real costuma ter **gap de 1 dia**; ⚠ com gap grande tende a ser deslistagem coincidente (ex.: não confunda o predecessor saudável de gap 1d com um ⚠ de gap 6d).

In [3]:
if DETECTAR_LINHAGEM:
    fim = {t: r['ultimo'] for t, r in RESUMO.items() if pd.notna(r['ultimo'])}
    print("Candidatos a predecessor:\n")
    for t in universo:
        r = RESUMO.get(t)
        if r is None or pd.isna(r['primeiro']) or r['primeiro'] <= pd.Timestamp(ANO_INI, 1, 31):
            continue
        for cand, f in fim.items():
            if cand != t and 1 <= (r['primeiro'] - f).days <= CONTINUIDADE_DIAS:
                rc = RESUMO[cand]
                penny = any(v < PISO_PRECO for v in rc['med_por_ano'].values())
                marca = " ⚠ (foi penny/RJ)" if (penny or rc['anos_sit']) else ""
                print(f"  {t} <- {cand} (gap {(r['primeiro']-f).days}d){marca}")
else:
    print("Detecção desativada.")

Candidatos a predecessor:

  ABEV3 <- AMBV3 (gap 3d)
  ABEV3 <- AMBV4 (gap 3d)
  ABEV3 <- VINE3 (gap 7d)
  AMAR3 <- MARI3 (gap 3d)
  AMER3 <- BTOW3 (gap 3d)
  AXIA3 <- CPLE6 (gap 3d)
  AXIA3 <- ELET3 (gap 3d)
  AXIA3 <- ELET6 (gap 3d)
  AXIA3 <- EQPA6 (gap 5d)
  AXIA3 <- USIM6 (gap 3d)
  AXIA6 <- CPLE6 (gap 3d)
  AXIA6 <- ELET3 (gap 3d)
  AXIA6 <- ELET6 (gap 3d)
  AXIA6 <- EQPA6 (gap 5d)
  AXIA6 <- USIM6 (gap 3d)
  B3SA3 <- BVMF3 (gap 3d)
  CLSC4 <- CLSC6 (gap 1d)
  CLSC4 <- UOLL4 (gap 2d)
  CSUD3 <- CARD3 (gap 1d)
  CSUD3 <- SHUL3 (gap 3d)
  DXCO3 <- DTEX3 (gap 1d)
  DXCO3 <- VVAR3 (gap 6d)
  EGIE3 <- TBLE3 (gap 1d)
  EMBJ3 <- EMBR3 (gap 3d)
  ENEV3 <- MPXE3 (gap 1d)
  FICT3 <- ATOM3 (gap 1d) ⚠ (foi penny/RJ)
  GOLL54 <- GOLL4 (gap 1d)
  GOLL54 <- JBSS3 (gap 6d)
  ISAE4 <- TRPL3 (gap 4d)
  ISAE4 <- TRPL4 (gap 4d)
  LAND3 <- TESA3 (gap 3d)
  MBRF3 <- BRFS3 (gap 1d)
  MBRF3 <- EQPA7 (gap 6d)
  MBRF3 <- MRFG3 (gap 1d)
  MOTV3 <- CCRO3 (gap 2d)
  NEXP3 <- BBRK3 (gap 1d) ⚠ (foi penny/RJ)
 

## 3. Classificação (v2)

In [4]:
# corte de liquidez: percentil da mediana de volume do universo (só nomes com dado)
adtv_univ = [RESUMO[t]['volmed_global'] for t in universo if t in RESUMO and pd.notna(RESUMO[t]['volmed_global'])]
CORTE_ADTV = float(np.percentile(adtv_univ, PERCENTIL_ADTV))
print(f"Corte de ADTV (p{PERCENTIL_ADTV} do universo) = R$ {CORTE_ADTV:,.0f}/dia\n")

def combinar(cods, chave):
    out = {}
    for c in cods:
        for a, v in RESUMO[c][chave].items(): out[a] = v if a not in out else (min(out[a], v) if chave=='med_por_ano' else v)
    return out

linhas = []
for t in universo:
    cods = [t] + MAPA_LINHAGEM.get(t, [])
    pres = [c for c in cods if c in RESUMO]
    if not pres:
        linhas.append(dict(ticker=t, decisao='SEM DADOS', motivo='sem série no COTAHIST', adtv_global=pd.NA,
                           anos_penny='', anos_situacao='', codigos=';'.join(cods))); continue

    med = combinar(pres, 'med_por_ano'); vol = combinar(pres, 'volmed_por_ano')
    sit = sorted({a for c in pres for a in RESUMO[c]['anos_sit']})
    adtv_global = float(np.nanmedian([RESUMO[c]['volmed_global'] for c in pres]))

    if t in REORG_TICKERS:
        anos_p = sorted(a for a, v in med.items() if v < PISO_PRECO)
        det = []
        if anos_p: det.append(f"penny {anos_p}")
        if sit:    det.append(f"RJ {sit}")
        corr = f" [corrobora: {', '.join(det)}]" if det else ""
        linhas.append(dict(ticker=t, decisao='EXCLUIR',
                           motivo=f"reorganização societária (identidade rompida; ISIN/ticker novo){corr}",
                           adtv_global=round(adtv_global, 0), anos_penny=','.join(map(str, anos_p)),
                           anos_situacao=','.join(map(str, sit)), codigos=';'.join(pres))); continue

    # identidade contínua -> janela recente
    anos_penny_rec = [a for a in ANOS_RECENTES if med.get(a, 9e9) < PISO_PRECO]
    adtv_penny = float(np.nanmedian([vol.get(a, adtv_global) for a in anos_penny_rec])) if anos_penny_rec else np.nan
    penny_excl = bool(anos_penny_rec) and (adtv_penny < CORTE_ADTV)
    rj_rec = [a for a in ANOS_RECENTES if a in sit]

    motivos = []
    if penny_excl: motivos.append(f"penny recente (mediana<R${PISO_PRECO:.0f}) e ilíquida em {anos_penny_rec} "
                                  f"(ADTV R$ {adtv_penny:,.0f} < corte R$ {CORTE_ADTV:,.0f})")
    if rj_rec:     motivos.append(f"situação especial recente em {rj_rec}")

    linhas.append(dict(ticker=t, decisao='EXCLUIR' if motivos else 'MANTER', motivo='; '.join(motivos),
                       adtv_global=round(adtv_global, 0),
                       anos_penny=','.join(map(str, sorted(a for a, v in med.items() if v < PISO_PRECO))),
                       anos_situacao=','.join(map(str, sit)), codigos=';'.join(pres)))

classif = pd.DataFrame(linhas).sort_values(['decisao', 'ticker']).reset_index(drop=True)
print(classif['decisao'].value_counts().to_string())

Corte de ADTV (p20 do universo) = R$ 829,107/dia

decisao
MANTER     102
EXCLUIR     16


## 4. Diagnóstico de fronteira e saídas

A tabela de diagnóstico mostra, para os nomes com penny antigo OU situação especial, por que ficaram ou saíram — útil para calibrar `PERCENTIL_ADTV` vendo a separação real de liquidez. Nada é mantido silenciosamente: `SEM DADOS` vai para revisão.

In [5]:
susp = classif[(classif['anos_penny'] != '') | (classif['anos_situacao'] != '')]
print("=== DIAGNÓSTICO (nomes com penny e/ou situação especial) ===")
print(susp[['ticker', 'decisao', 'adtv_global', 'anos_penny', 'anos_situacao', 'motivo']]
      .to_string(index=False), "\n")

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
classif.to_csv(PASTA_SAIDA / "classificacao_universo_v2.csv", index=False, encoding="utf-8-sig")
(classif[classif['decisao'] == 'EXCLUIR'][['ticker', 'motivo', 'anos_penny', 'anos_situacao', 'codigos']]
    .to_csv(PASTA_SAIDA / "tickers_excluidos_integridade.csv", index=False, encoding="utf-8-sig"))
(classif[classif['decisao'] == 'MANTER'][['ticker']]
    .to_csv(PASTA_SAIDA / "tickers_finais_v2.csv", index=False, encoding="utf-8-sig"))

print("=== RESUMO ===")
for d in ['MANTER', 'EXCLUIR', 'SEM DADOS']:
    print(f"  {d}: {(classif['decisao'] == d).sum()}")
print("\n--- EXCLUÍDOS ---")
for _, r in classif[classif['decisao'] == 'EXCLUIR'].iterrows():
    print(f"  {r['ticker']:8s} | {r['motivo']}")
sem = classif[classif['decisao'] == 'SEM DADOS']['ticker'].tolist()
if sem: print("\n--- SEM DADOS (revisar) ---\n ", ", ".join(sem))
print(f"\nGravado em {PASTA_SAIDA}: classificacao_universo_v2.csv, tickers_excluidos_integridade.csv, tickers_finais_v2.csv")

=== DIAGNÓSTICO (nomes com penny e/ou situação especial) ===


ticker decisao  adtv_global                    anos_penny                                                    anos_situacao                                                                                                                 motivo
 AMER3 EXCLUIR   24193775.0                          2024                                                   2023,2024,2025                                                                        situação especial recente em [2023, 2024, 2025]
 ETER3 EXCLUIR     825536.0                          2018                               2018,2019,2020,2021,2022,2023,2024                                                                              situação especial recente em [2023, 2024]
 FICT3 EXCLUIR      77491.0                     2015,2016                                                   2015,2016,2017 reorganização societária (identidade rompida; ISIN/ticker novo) [corrobora: penny [2015, 2016], RJ [2015, 2016, 2017]]
GOLL54 EXCLUIR   16235163.0     

# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no Apêndice Classificador de Integridade |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Explica a lógica de auditoria point-in-time usando dados brutos do COTAHIST (B3). |
| 2 | Documentar o processo | ✅ | Define os três pilares objetivos de exclusão (reorganizações, CODBDI especial, penny stocks). |
| 3 | Divisão clara de células | ✅ | Divisão em células de carga, análise por ativo, e consolidação de exportação. |
| 4 | Modularizar código | ✅ | Modularizado em rotinas de leitura de lote de arquivos zip/csv da B3. |
| 5 | Registrar dependências | ✅ | Utiliza pandas e numpy definidos em requirements. |
| 6 | Controle de versão | ✅ | Sob controle de versão Git. |
| 7 | Construir um pipeline | ✅ | Integrado no fluxo de integridade da Etapa 3, exportando tickers_excluidos_integridade.csv. |
| 8 | Compartilhar/explicar dados | ✅ | Indica o download dos dados públicos históricos do COTAHIST (B3). |
| 9 | Ler, rodar e explorar | ✅ | Executa de ponta a ponta gerando o mesmo resumo de 102 mantidos / 16 excluídos. |
| 10 | Pesquisa aberta | ✅ | Metodologia de auditoria publicada livremente. |
